# CAES mini-project: system design and simulation

Compressed Air Energy Storage (CAES) stores electricity by compressing air into a reservoir. During discharge, the air is heated and expanded through a turbine to produce electricity. The project compares how compression heat management and reservoir design affect performance.

## System variants

The six variants combine three architectures with two reservoir types:

| Architecture | Constant-volume reservoir | Constant-pressure reservoir |
|---|---|---|
| Single-stage compression and expansion; compression heat rejected | 1a | 1b |
| Single-stage with thermal energy storage (TES) | 2a | 2b |
| Two-stage compression with intercooling and two-stage expansion with reheat; no TES | 3a | 3b |

A constant-volume reservoir operates between 16 bar and a minimum pressure of 8 bar, so its pressure falls during discharge. A constant-pressure reservoir remains at 16 bar while its volume varies. Variants 2a and 2b store compression heat in a hot-water TES and reuse it to preheat discharge air before combustion. Variants 3a and 3b use equal pressure-ratio stages; intercooling heat is rejected, and fuel reheats the air before each turbine stage.

## Common specification

- Working fluid: real-gas air (CoolProp).
- Ambient conditions: 15°C and 1.013 bar.
- Maximum storage pressure: 16 bar; minimum pressure for constant-volume storage: 8 bar.
- Target output: 1 MW at the turbine/generator shaft for 4 hours (4 MWh per cycle); charge duration is also 4 hours.
- Compressor isentropic efficiency: 0.85; turbine isentropic efficiency: 0.88.
- Mechanical/generator efficiency: 0.98.
- Turbine inlet temperature: 550°C. Combustion is represented as heat addition to reach this temperature, without modelling combustion chemistry.
- Cooler and TES return pinches: 10 K. Air is cooled towards ambient temperature plus the cooler pinch before storage.
- The reservoir is treated as a uniform 0D store. Stored air is assumed to return to ambient temperature before discharge and remain near ambient during discharge.

## Performance and design tasks

Use the same two-step workflow for the selected variant:

1. **Design:** size the required components, heat exchangers, and storage volume for the common output target.
2. **Simulation:** model a complete charge/discharge cycle and evaluate performance.

The common round-trip metric values fuel at its electrical equivalent:

\[
\eta = \frac{W_{\mathrm{turbine}}}{W_{\mathrm{compressor}} + \eta_f Q_{\mathrm{combustion}}},
\qquad \eta_f = 0.40.
\]

Compare round-trip efficiency, storage volume, heat-exchanger areas, maximum temperatures, and the electrical/fuel input split. Include energy (and optionally exergy) balances, identify major losses, and quantify at least one proposed performance improvement. The final comparison should show how thermal storage, staging, and reservoir type affect these results.

In [1]:
# --- imports ---
from pathlib import Path
import sys
''''''
for root in (Path.cwd(), *Path.cwd().parents):
    for package_root in (root, root / "caldis-0.3.0"):
        if (package_root / "caldis" / "__init__.py").is_file():
            sys.path.insert(0, str(package_root))
            break
    else:
        continue
    break
''''''
import numpy as np
import caldis
from CoolProp.CoolProp import PropsSI
from caldis.core.system import System
from caldis.components import FluidBoundary
from caldis.components import Compressor
from caldis.fluids import CoolPropBackend
from caldis.components import SizingHX
from caldis.components import CombustionChamber
from caldis.components import Turbine
from caldis.solvers import solve  


Initilize paramaters

Ambient Air --> Compressor --> Cooler --> Air Storage --> Combustor --> Turbine --> Exhuast

In [2]:
# --- system definition ---
#Inlet Compressor 
T0=15 +273.15 #k
P0=1.013e5
Eta_ISC=0.85

#Cooler (water)
Tpinch=10 #k


#Air Storage
Pmax=16e5
    #Volume is Variable probably use constraint to find volume based on mass flow rate and time of storage

#Combustion Chamber
Ttit=550 +273.15 #k
Eta_Fuel=0.4

#Turbine
Eta_IST=0.85
Eta_Mech=0.98

We need to define Boundaries

In [ ]:
be_air= CoolPropBackend("Air")
be_water= CoolPropBackend("Water")

#h0=PropsSI("H","T",T0,"P",P0,"Air")

#The mass flowrate was assumed to be -1 (going outwards) as the model requires a mass flowrate to be defined. 
#The actual mass flowrate will be calculated by the solver based on the system constraints.
Source = FluidBoundary(name="Source", p=P0, T=T0, h=None, mdot=-1, backend=be_air)

# The pressure outlet of the compresor is the same asd the maximum one as the heat exchanger 
# does not change the pressure of the fluid. The heat exchanger is only used to cool down the 
# air before it enters the storage tank.

#So we have P0 Pmax T0 3 components

Comp = Compressor(name="Comp", backend=be_air, eta_is=0.85)
# we need to find assumptions of H and area ratio for sizing the heat exchanger. For now, we will leave them as None and let the user define them later.
#Hex = SizingHX(name="Hex", backend_hot=be_air, backend_cold=be_water, h_hot=None, h_cold=None, area_ratio_cold_to_hot=None)

Sink = FluidBoundary(name="Sink", p=Pmax, T=None, h=None, mdot=None, backend=be_air)


In [4]:
sys_=System("Comp_Parts")

sys_.add(Source, Comp, Sink)
sys_.connect(Source.port, Comp.inl)
sys_.connect(Comp.out, Sink.port)
sol = solve(sys_)


✓ DOF: square (12 unknowns)
✓ start point: evaluable
⚠ 1 start-sanity warning(s):
   - Comp: p_out (1.00 bar) must be > p_in (1.00 bar) for compression
  (non-blocking.)
✓ SOLVED (|F_scaled|=3.3e-09)


In [5]:
Comp.describe()


=== Compressor('Comp') ===
  ports (p [bar], mdot [kg/s], T [°C]):
    inl        p=1.013  mdot=+1  h=414.4kJ/kg  T=15.00
    out        p=16.000  mdot=-1  h=822.4kJ/kg  T=410.60
  outputs:
    Wdot             = -408069
    pressure_ratio   = 15.7947


{'variables': {},
 'ports': {'inl': {'p': 101299.99999566667,
   'mdot': 0.9999999967000007,
   'h': 414374.631759745},
  'out': {'p': 1599999.9949999985,
   'mdot': -0.999999996633334,
   'h': 822443.6583456455}},
 'outputs': {'Wdot': -408069.02523927303,
  'pressure_ratio': 15.794669250428845}}